In [ ]:
import sys; sys.path.insert(0, "..")
import matplotlib
matplotlib.use("Agg")
import pyulog
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation as R_
from pathlib import Path
from pid_optimizer.plant_model import PlantModel
from pid_optimizer.gains import Gains, GAIN_BOUNDS, HERMIT_REFERENCE_GAINS
from pid_optimizer.pid_simulator import PX4Simulator, compute_rmse
from pid_optimizer.ga import GeneticAlgorithm

PLANT_PATH   = "../artifacts/plant_model.json"
CHECKPOINT   = "../artifacts/ga_checkpoint.json"
BEST_GAINS   = "../artifacts/best_gains.json"
LOG_PATH     = "../px4_logs/Hermit/testes_PID_position_31-08/log_12_2024-8-30-16-21-54.ulg"

plant = PlantModel.load(PLANT_PATH)

In [ ]:
log = pyulog.ULog(LOG_PATH, disable_str_exceptions=True)
t0 = log.start_timestamp

def to_df(log, topic, multi_id=0):
    d = next(d for d in log.data_list if d.name == topic and d.multi_id == multi_id)
    df = pd.DataFrame({f: d.data[f] for f in d.data})
    df["t_s"] = (df["timestamp"] - t0) / 1e6
    return df.sort_values("t_s").reset_index(drop=True)

DT, T_MAX = 0.004, 20.0
t_sim = np.arange(0, T_MAX, DT)

pos_sp_df = to_df(log, "vehicle_local_position_setpoint")
att_sp_df = to_df(log, "vehicle_attitude_setpoint")
pos_df    = to_df(log, "vehicle_local_position")

def interp_col(src_df, col, t_out):
    mask = ~src_df[col].isna()
    if mask.sum() < 2:
        return np.zeros(len(t_out))
    return np.interp(t_out, src_df["t_s"][mask].values, src_df[col][mask].values)

q_sp = att_sp_df[["q_d[0]","q_d[1]","q_d[2]","q_d[3]"]].values
yaw_sp = R_.from_quat(q_sp[:, [1,2,3,0]]).as_euler("xyz")[:, 2]

setpoints = pd.DataFrame({
    "x": interp_col(pos_sp_df, "x", t_sim), "y": interp_col(pos_sp_df, "y", t_sim),
    "z": interp_col(pos_sp_df, "z", t_sim), "vx": interp_col(pos_sp_df, "vx", t_sim),
    "vy": interp_col(pos_sp_df, "vy", t_sim), "vz": interp_col(pos_sp_df, "vz", t_sim),
    "roll": float('nan'), "pitch": float('nan'),
    "yaw": np.interp(t_sim, att_sp_df["t_s"].values, yaw_sp),
})

actual_pos = pd.DataFrame({
    "x": interp_col(pos_df, "x", t_sim),
    "y": interp_col(pos_df, "y", t_sim),
    "z": interp_col(pos_df, "z", t_sim),
})

ref_sim = PX4Simulator(plant, HERMIT_REFERENCE_GAINS).run(setpoints, dt=DT)
baseline_rmse = compute_rmse(ref_sim[["x","y","z"]], actual_pos, cols=["x","y","z"])
print("Baseline RMSE (reference gains):", {k: f"{v:.4f}" for k, v in baseline_rmse.items()})

In [ ]:
WEIGHTS = {"x": 1.5, "y": 1.5, "z": 2.0}

def fitness(gains_array: np.ndarray) -> float:
    g = Gains.from_array(gains_array)
    try:
        result = PX4Simulator(plant, g).run(setpoints, dt=DT)
        rmse = compute_rmse(result[["x","y","z"]], actual_pos, cols=["x","y","z"])
        score = sum(WEIGHTS[k] * rmse[k] for k in WEIGHTS)
        if not np.isfinite(score): return 1e6
        return float(score)
    except Exception:
        return 1e6

ref_score = fitness(HERMIT_REFERENCE_GAINS.to_array())
print(f"Reference gains fitness: {ref_score:.4f}")

In [ ]:
import os, json

GENERATIONS = 5  # Set to 200 for real optimization
CHECKPOINT_EVERY = 5

if os.path.exists(CHECKPOINT):
    ga = GeneticAlgorithm(GAIN_BOUNDS, fitness_fn=fitness, pop_size=20, seed=42)
    ga.load_checkpoint(CHECKPOINT)
    print(f"Resumed from generation {ga.generation}, best fitness={ga.best_fitness:.4f}")
else:
    ga = GeneticAlgorithm(GAIN_BOUNDS, fitness_fn=fitness, pop_size=20, seed=42)
    ga.initialize()
    print(f"Initialized. Generation 0 best fitness={ga.best_fitness:.4f}")

remaining = GENERATIONS - ga.generation
print(f"Running {remaining} more generations...")
for gen in range(remaining):
    ga.step()
    if ga.generation % CHECKPOINT_EVERY == 0:
        ga.save_checkpoint(CHECKPOINT)
        print(f"Gen {ga.generation:4d} | best={ga.best_fitness:.4f} | mean={ga.history[-1]['mean']:.4f}")

scores_argsort = ga.scores.argsort()
best_n = [{"gains": ga.population[i].tolist(), "fitness": float(ga.scores[i])}
          for i in scores_argsort[:5]]
with open(BEST_GAINS, "w") as f:
    json.dump({"reference_fitness": ref_score, "candidates": best_n}, f, indent=2)
print(f"\nBest fitness: {ga.best_fitness:.4f}  (reference: {ref_score:.4f})")
print(f"Saved to {BEST_GAINS}")

In [ ]:
if ga.history:
    gens  = [h["generation"] for h in ga.history]
    bests = [h["best"]  for h in ga.history]
    means = [h["mean"]  for h in ga.history]
    stds  = [h["std"]   for h in ga.history]

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(gens, bests, label="best", linewidth=2)
    ax.plot(gens, means, label="mean", alpha=0.7)
    ax.fill_between(gens, np.array(means)-np.array(stds),
                    np.array(means)+np.array(stds), alpha=0.2, label="\u00b11\u03c3")
    ax.axhline(ref_score, color="red", linestyle="--", label="reference")
    ax.set_xlabel("Generation"); ax.set_ylabel("Fitness (weighted RMSE)")
    ax.set_title("GA Convergence"); ax.legend()
    plt.tight_layout(); plt.savefig("/tmp/ga_convergence.png"); plt.close()
    print("Saved convergence plot to /tmp/ga_convergence.png")
else:
    print("No history to plot")

In [ ]:
from dataclasses import fields as dc_fields

gain_names = [f.name for f in dc_fields(Gains)]
lo = np.array([b[0] for b in GAIN_BOUNDS])
hi = np.array([b[1] for b in GAIN_BOUNDS])
top10 = ga.population[ga.scores.argsort()[:min(10, len(ga.population))]]
normalized = (top10 - lo) / (hi - lo + 1e-12)

fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(normalized.T, aspect="auto", cmap="coolwarm", vmin=0, vmax=1)
ax.set_yticks(range(len(gain_names))); ax.set_yticklabels(gain_names, fontsize=8)
ax.set_xlabel("Top individuals"); ax.set_title("Gain values (normalized to bounds)")
plt.colorbar(im, ax=ax, label="0=low bound, 1=high bound")
plt.tight_layout(); plt.savefig("/tmp/ga_heatmap.png"); plt.close()
print("Saved heatmap to /tmp/ga_heatmap.png")